# M³Diff — Notebook 1: Bộ test tổng hợp có kiểm soát

Tự sinh cặp ảnh bằng xử lý ảnh (biết chính xác thay đổi gì, ở đâu, mức độ bao nhiêu), chạy M³Diff, chấm câu model sinh ra.

| | |
|---|---|
| Thiết kế | 6 loại thay đổi × 3 mức × 30 cặp + 2 nhóm đối chứng × 30 cặp = **600 cặp** |
| Ground truth vị trí | các pixel mà A ≠ B, tính trên ảnh chưa nén (dùng ở Notebook 2, 3) |
| Chấm câu | đúng loại / đúng vật / đúng cả hai + tỉ lệ báo thay đổi giả ở đối chứng |
| Thời gian | ~15 phút chuẩn bị + ~30 phút GPU |

**Trước khi chạy:** Compute profile **A10**, CPU 4, RAM **16 GiB**; Volume `m3diff` gắn ở `/mnt/m3diff`.
Chạy lần lượt từ trên xuống. **Không đổi cấu hình máy khi đang chạy** (đổi là máy bị dựng lại).
Dữ liệu và kết quả lưu ở `/mnt/m3diff/synth/` (trên Volume, không mất khi tắt máy).

## Bước 1 — Cấu hình và kiểm tra

In [ ]:
import os, sys, json, shutil, subprocess

VOL        = "/mnt/m3diff"                       # Volume tạo ở Files panel
REPO       = "/root/M3Diff"
VENV       = "/root/m3diff-venv"
PY         = f"{VENV}/bin/python"
CKPT       = "/root/ckpt/M3Diff"                 # checkpoint 16GB, tải lại mỗi phiên (nhanh)
RESULTS    = f"{VOL}/results/OmniDiff_Real"

# A10 / L4 / A100 (>= 24GB): QUANT="none", DTYPE="bfloat16"  -> đúng bản gốc
# T4 (16GB):                 QUANT="8bit", DTYPE="float16"
QUANT = "none"
DTYPE = "bfloat16"

# Chỉ cần nếu tải Hugging Face báo lỗi 401/403: dán token vào đây
HF_TOKEN = os.environ.get("HF_TOKEN", "")

!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir(VOL):
    print(f"\n!!! Chưa thấy Volume {VOL}. Vào Files -> tạo Volume tên m3diff. Hiện /mnt có:", os.listdir("/mnt") if os.path.isdir("/mnt") else "(trống)")
else:
    os.makedirs(RESULTS, exist_ok=True)
    print("\nVolume OK:", VOL)

## Bước 2 — Tạo môi trường Python 3.10 riêng + cài thư viện
Kernel của Modal là Python 3.12 với torch bản mới. Repo cần đúng torch 2.2.2 / transformers 4.40.0, nên tạo venv riêng bằng `uv` (không đụng vào kernel, không phải restart).

In [ ]:
%uv pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}
if not os.path.exists(PY):
    !{UV} venv {VENV} --python 3.10 -q

%cd {REPO}
!{UV} pip install -q --python {PY} -e ".[eval]" "numpy<2" accelerate==0.29.1 bitsandbytes==0.43.3 install-jdk
!{PY} -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__); print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')"

Kỳ vọng: `torch 2.2.2+cu121`, `transformers 4.40.0`, `numpy 1.26.4`, `CUDA: True | NVIDIA A10 | ~22 GB`.

## Bước 3 — Tải checkpoint M³Diff (16GB) và ảnh test OmniDiff (Real + Render, dùng làm ảnh nền)

In [ ]:
dl = f'''
from huggingface_hub import snapshot_download
tok = {HF_TOKEN!r} or None
snapshot_download("IVC-liuyuan/M3Diff", local_dir="{CKPT}", token=tok)
snapshot_download("IVC-liuyuan/OmniDiff", repo_type="dataset", local_dir="{REPO}/datasets/OmniDiff",
                  allow_patterns=["Real/test/*", "Render/test/*"], token=tok)
print("Tải xong")
'''
subprocess.run([PY, "-c", dl], check=True)

questions = []
for split in ["Real", "Render"]:
    with open(f"{REPO}/data/evaluation/prompt/OmniDiff_{split}_test.json") as f:
        questions += json.load(f)
needed = sorted({img for q in questions for img in q["image"]})
missing = [p for p in needed if not os.path.exists(f"{REPO}/datasets/OmniDiff/{p}")]
!du -sh {CKPT}
print(f"Ảnh cần: {len(needed)} | còn thiếu: {len(missing)}", missing[:10])

Phải ra checkpoint khoảng `16G` và `còn thiếu: 0`.

## Bước 4 — Patch script eval
1. Đặt toàn bộ model lên GPU (tránh lỗi `Cannot copy out of meta tensor`); nếu `QUANT` là 8bit/4bit thì nén phần LLM.
2. Chạy tiếp được: bỏ qua câu đã có trong file kết quả, ghi file sau từng câu.

Chạy lại nhiều lần không sao.

In [ ]:
path = f"{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py"
src = open(path).read()

# --- Patch 1: nén model cho vừa T4 ---
old1 = """    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
    )"""
new1 = """    quant_mode = os.environ.get("M3DIFF_QUANT", "8bit")
    skip = ["vision_tower", "vision_resampler", "mm_projector", "lm_head"]
    extra = {}
    if quant_mode == "8bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_8bit=True, llm_int8_skip_modules=skip)
    elif quant_mode == "4bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=skip)
    print(f"[patch] quant_mode={quant_mode}", flush=True)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
        device_map={"": 0},
        **extra,
    )"""

# --- Patch 2: chạy tiếp từ chỗ dừng + ghi từng câu ---
old2 = '    ans_file = open(answers_file, "w")\n'
new2 = """    done_lines = []
    if os.path.exists(answers_file):
        for l in open(answers_file):
            try:
                json.loads(l)
                done_lines.append(l.strip())
            except Exception:
                pass
    done_ids = {json.loads(l)["id"] for l in done_lines}
    with open(answers_file, "w") as f:
        f.write("".join(l + "\\n" for l in done_lines))
    questions = [q for q in questions if q["id"] not in done_ids]
    print(f"[patch] resume: {len(done_ids)} done, {len(questions)} left", flush=True)
    ans_file = open(answers_file, "a")
"""
old3 = '        ans_file.flush()\n'
new3 = '        ans_file.close()\n        ans_file = open(answers_file, "a")\n'

if "M3DIFF_QUANT" not in src:
    assert old1 in src, "Không tìm thấy đoạn patch 1"
    src = src.replace(old1, new1)
if "[patch] resume" not in src:
    assert old2 in src and old3 in src, "Không tìm thấy đoạn patch 2"
    src = src.replace(old2, new2).replace(old3, new3)
open(path, "w").write(src)
print("Patch OK:", "M3DIFF_QUANT" in src, "[patch] resume" in src)

## Bước 5 — Cài thêm thư viện và ghi các script
Script được ghi lên Volume (`/mnt/m3diff/synth/code/`) để Notebook 2 và 3 dùng lại đúng các hàm này
(đặc biệt hàm chuyển toạ độ lưới 27×27 ↔ ảnh gốc, tránh lệch toạ độ giữa các notebook).

In [ ]:
!{UV} pip install -q --python {PY} "opencv-python-headless<4.12" "numpy<2" pandas matplotlib pycocotools
os.environ["MPLBACKEND"] = "Agg"
SYN  = f"{VOL}/synth"
CODE = f"{SYN}/code"
DATA = f"{SYN}/data"
RES  = f"{SYN}/results"
for d in [CODE, DATA, RES]:
    os.makedirs(d, exist_ok=True)
QFILES = f"{REPO}/data/evaluation/prompt/OmniDiff_Real_test.json {REPO}/data/evaluation/prompt/OmniDiff_Render_test.json"
!{PY} -c "import cv2, numpy, pandas, pycocotools; print('cv2', cv2.__version__, '| numpy', numpy.__version__)"

Kỳ vọng `numpy 1.26.x` (phải < 2, nếu không model sẽ lỗi).

**synth_common.py** — Hàm dùng chung: toạ độ pad, đọc dữ liệu, khoảng tin cậy

In [ ]:
%%writefile /mnt/m3diff/synth/code/synth_common.py
"""Hàm dùng chung cho 3 notebook của bộ test tổng hợp.

Hệ toạ độ thống nhất: MỌI so sánh heatmap với ground truth đều làm trên hệ toạ độ ẢNH GỐC (W×H).
Model (LLaVA, image_aspect_ratio='pad') đặt ảnh vào giữa một hình vuông cạnh S = max(W, H),
rồi resize về 384×384 → lưới 27×27 patch (mỗi patch 14 px). Các hàm dưới đây chuyển qua lại
giữa lưới 27×27 và ảnh gốc, có bỏ phần viền pad.
"""
import json
import os

os.environ["MPLBACKEND"] = "Agg"      # kernel Jupyter truyền backend inline sang, venv không có -> ép Agg
import cv2
import numpy as np

GRID = 27

TYPES = ["add", "remove", "color", "move", "size", "light"]
CONTROLS = ["same", "jpeg"]
OBJECT_TYPES = ["add", "remove", "color", "move", "size"]   # các loại có vật thể cụ thể
LOCALIZABLE = OBJECT_TYPES                                  # 'light' thay đổi toàn ảnh -> loại khỏi phần định vị

TYPE_VI = {"add": "Thêm vật", "remove": "Xoá vật", "color": "Đổi màu", "move": "Dời vật",
           "size": "Đổi kích thước", "light": "Đổi sáng", "same": "Đối chứng A == B",
           "jpeg": "Đối chứng B = JPEG(A)"}


# ---------- hệ toạ độ pad ----------
def pad_geometry(w, h):
    """Cạnh hình vuông S và vị trí (x0, y0) của ảnh gốc trong hình vuông (giống expand2square)."""
    s = max(w, h)
    return s, (s - w) // 2, (s - h) // 2


def valid_grid_box(w, h):
    """Vùng lưới 27×27 chứa ảnh thật (toạ độ ô, dạng float): (gx0, gy0, gx1, gy1)."""
    s, x0, y0 = pad_geometry(w, h)
    return x0 / s * GRID, y0 / s * GRID, (x0 + w) / s * GRID, (y0 + h) / s * GRID


def grid_to_image(heat, w, h):
    """Heatmap 27×27 (trên hình vuông đã pad) -> heatmap W×H trên ảnh gốc (bỏ viền pad)."""
    s, x0, y0 = pad_geometry(w, h)
    up = cv2.resize(np.asarray(heat, np.float32), (s, s), interpolation=cv2.INTER_LINEAR)
    return up[y0:y0 + h, x0:x0 + w]


def image_to_grid(mask, grid=GRID):
    """Mặt nạ W×H trên ảnh gốc -> tỉ lệ phủ của từng ô 27×27 (0..1), cùng quy tắc pad với model."""
    h, w = mask.shape[:2]
    s, x0, y0 = pad_geometry(w, h)
    sq = np.zeros((s, s), np.float32)
    sq[y0:y0 + h, x0:x0 + w] = (mask > 0).astype(np.float32)
    return cv2.resize(sq, (grid, grid), interpolation=cv2.INTER_AREA)


# ---------- đọc dữ liệu ----------
def load_manifest(root):
    with open(os.path.join(root, "manifest.jsonl")) as f:
        return [json.loads(l) for l in f if l.strip()]


def load_gt(root, rec):
    m = cv2.imread(os.path.join(root, rec["gt_mask"]), cv2.IMREAD_GRAYSCALE)
    return (m > 127) if m is not None else None


def load_pair(root, rec):
    a = cv2.imread(os.path.join(root, rec["image_A"]), cv2.IMREAD_COLOR)
    b = cv2.imread(os.path.join(root, rec["image_B"]), cv2.IMREAD_COLOR)
    return a, b


# ---------- thống kê ----------
def wilson(k, n, z=1.96):
    """Khoảng tin cậy 95% Wilson cho tỉ lệ k/n. Trả về (p, lo, hi)."""
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - r), min(1.0, c + r)


def bootstrap_ci(values, n_boot=2000, seed=0, stat=np.mean):
    v = np.asarray([x for x in values if np.isfinite(x)], float)
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    bs = [stat(v[rng.integers(0, len(v), len(v))]) for _ in range(n_boot)]
    return float(stat(v)), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

**prep_coco_objects.py** — Lấy vật thể có mặt nạ từ COCO val2017

In [ ]:
%%writefile /mnt/m3diff/synth/code/prep_coco_objects.py
"""Lấy vật thể đã có mặt nạ từ COCO val2017 -> ảnh cắt nền RGBA + objects.csv.

Lọc để vật thể "sạch": không phải đám đông, đủ lớn, không chạm mép ảnh, một mảnh duy nhất,
không bị vật khác che (vật khác chiếm >15% khung bao của nó thì bỏ).
"""
import argparse, io, json, os, time, urllib.request, zipfile
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
import pandas as pd

CATS = ["bottle", "cup", "bowl", "vase", "chair", "potted plant", "teddy bear", "backpack",
        "umbrella", "suitcase", "laptop", "book", "clock", "dog", "cat", "bicycle", "car",
        "apple", "banana", "handbag"]
ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMG_URL = "http://images.cocodataset.org/val2017/{}"
SUPPORT = {"dining table", "bed", "couch", "bench"}   # mặt phẳng đỡ vật, không tính là vật che


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--cache", required=True, help="thư mục lưu annotations + ảnh COCO tải về")
    p.add_argument("--out", required=True, help="thư mục lưu vật thể RGBA + objects.csv")
    p.add_argument("--per-cat", type=int, default=15)
    p.add_argument("--min-area", type=int, default=6000)
    p.add_argument("--seed", type=int, default=2026)
    p.add_argument("--ann-json", default=None, help="(test) đường dẫn instances_val2017.json có sẵn")
    return p.parse_args()


def fetch(url, path, tries=4):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for t in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                data = r.read()
            with open(path + ".part", "wb") as f:
                f.write(data)
            os.replace(path + ".part", path)
            return path
        except Exception as e:
            print(f"  lỗi tải {url} (lần {t+1}): {e}", flush=True)
            time.sleep(3 * (t + 1))
    raise RuntimeError(f"Không tải được {url}")


def get_ann_json(cache):
    js = os.path.join(cache, "instances_val2017.json")
    if os.path.exists(js):
        return js
    z = os.path.join(cache, "annotations_trainval2017.zip")
    print("Tải annotations COCO (~250MB)...", flush=True)
    fetch(ANN_URL, z)
    with zipfile.ZipFile(z) as zf:
        with zf.open("annotations/instances_val2017.json") as src, open(js, "wb") as dst:
            dst.write(src.read())
    os.remove(z)
    return js


def sheet(df, out, n=40):
    """Ảnh tổng hợp các vật đã cắt nền (nền ô cờ để thấy rõ mép) để kiểm tra bằng mắt."""
    import matplotlib.pyplot as plt
    sub = df.sample(min(n, len(df)), random_state=0) if len(df) else df
    cols = 10
    rows = max(1, int(np.ceil(len(sub) / cols)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 2.0), facecolor="#fcfcfb", squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, sub.iterrows()):
        im = cv2.imread(os.path.join(out, r["file"]), cv2.IMREAD_UNCHANGED)
        h, w = im.shape[:2]
        yy, xx = np.mgrid[0:h, 0:w]
        chk = np.where(((yy // 12 + xx // 12) % 2)[..., None] == 0, 220, 180).astype(np.float32)
        al = im[..., 3:4] / 255.0
        comp = (im[..., :3] * al + chk * (1 - al)).astype(np.uint8)
        ax.imshow(cv2.cvtColor(comp, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{r['category']} S={r['mean_sat']:.0f}", fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "objects_sheet.png"), dpi=90, facecolor="#fcfcfb")
    plt.close(fig)


def main():
    a = parse()
    os.makedirs(a.cache, exist_ok=True)
    os.makedirs(a.out, exist_ok=True)
    img_dir = os.path.join(a.cache, "val2017")
    os.makedirs(img_dir, exist_ok=True)

    from pycocotools.coco import COCO
    coco = COCO(a.ann_json or get_ann_json(a.cache))
    rng = np.random.default_rng(a.seed)
    support_ids = set(coco.getCatIds(catNms=list(SUPPORT)))
    rows = []
    for cat in CATS:
        cid = coco.getCatIds(catNms=[cat])
        if not cid:
            print("  không thấy lớp", cat); continue
        cid = cid[0]
        ann_ids = coco.getAnnIds(catIds=[cid], iscrowd=False)
        ann_ids = [ann_ids[i] for i in rng.permutation(len(ann_ids))]
        kept = 0
        for aid in ann_ids:
            if kept >= a.per_cat:
                break
            ann = coco.loadAnns(aid)[0]
            info = coco.loadImgs(ann["image_id"])[0]
            W, H = info["width"], info["height"]
            x, y, w, h = ann["bbox"]
            if ann["area"] < a.min_area or w < 60 or h < 60:
                continue
            if x <= 2 or y <= 2 or x + w >= W - 2 or y + h >= H - 2:
                continue                                   # chạm mép -> có thể bị cắt
            seg = ann["segmentation"]
            if not isinstance(seg, list) or len(seg) != 1:
                continue                                   # nhiều mảnh -> thường bị che
            if ann["area"] / (w * h) < 0.35:
                continue                                   # quá mảnh/rỗng
            m = coco.annToMask(ann).astype(bool)
            x0, y0, x1, y1 = int(x), int(y), int(np.ceil(x + w)), int(np.ceil(y + h))
            other = np.zeros_like(m)
            for o in coco.loadAnns(coco.getAnnIds(imgIds=[ann["image_id"]], iscrowd=None)):
                if o["id"] != aid and o["category_id"] not in support_ids:
                    other |= coco.annToMask(o).astype(bool)
            if (other[y0:y1, x0:x1] & ~m[y0:y1, x0:x1]).sum() > 0.15 * (x1 - x0) * (y1 - y0):
                continue                                   # vật khác chen vào khung -> có thể che
            ip = fetch(IMG_URL.format(info["file_name"]), os.path.join(img_dir, info["file_name"]))
            img = cv2.imread(ip, cv2.IMREAD_COLOR)
            if img is None:
                continue
            pad = 2
            cx0, cy0 = max(0, x0 - pad), max(0, y0 - pad)
            cx1, cy1 = min(W, x1 + pad), min(H, y1 + pad)
            crop = img[cy0:cy1, cx0:cx1]
            alpha = (m[cy0:cy1, cx0:cx1] * 255).astype(np.uint8)
            rgba = np.dstack([crop, alpha])
            fn = f"{cat.replace(' ', '_')}_{aid}.png"
            cv2.imwrite(os.path.join(a.out, fn), rgba)
            hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
            sat = float(hsv[..., 1][alpha > 127].mean())
            rows.append({"obj_id": int(aid), "category": cat, "file": fn, "w": cx1 - cx0, "h": cy1 - cy0,
                         "mask_area": int((alpha > 127).sum()), "mean_sat": round(sat, 1)})
            kept += 1
        print(f"  {cat:<13} {kept} vật", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(a.out, "objects.csv"), index=False)
    sheet(df, a.out)
    print(f"\nTổng {len(df)} vật thể, {int((df.mean_sat >= 70).sum())} vật đủ đậm màu cho loại 'đổi màu'")


if __name__ == "__main__":
    main()

**make_synth.py** — Sinh cặp ảnh tổng hợp + ground truth + manifest

In [ ]:
%%writefile /mnt/m3diff/synth/code/make_synth.py
"""Sinh bộ test tổng hợp có kiểm soát cho M3Diff.

6 loại thay đổi × 3 mức × N cặp + 2 nhóm đối chứng × N cặp.
Ground truth định vị: GT = các pixel mà A ≠ B, tính trên ảnh tổng hợp CHƯA NÉN (lưu PNG).
Riêng đối chứng JPEG: GT rỗng (không có thay đổi ngữ nghĩa), ảnh B lưu JPEG.
Mọi tham số của từng cặp (seed, nền, vật, vị trí, tỉ lệ, hue, gamma) được ghi vào manifest.jsonl.
"""
import argparse, json, os

import cv2
import numpy as np
import pandas as pd

LONG_SIDE = 768
FEATHER_SIGMA = 1.2
MARGIN = 8

LEVELS = {                      # 3 mức cho từng loại (từ nhỏ đến lớn)
    "add":    [0.01, 0.04, 0.10],     # diện tích vật / diện tích ảnh
    "remove": [0.01, 0.04, 0.10],
    "color":  [30, 60, 120],          # độ xoay Hue (độ)
    "move":   [0.10, 0.25, 0.50],     # quãng dời / chiều rộng ảnh
    "size":   [0.6, 1.5, 2.0],        # hệ số phóng theo chiều dài
    "light":  [1.25, 1.6, 2.2],       # gamma (hướng sáng/tối chọn ngẫu nhiên)
}
BASE_AREA = {"color": 0.04, "move": 0.03, "size": 0.03, "same": 0.04, "jpeg": 0.04}
JPEG_Q = 85
MIN_SAT_COLOR = 70               # vật cho loại 'đổi màu' phải đủ đậm màu (S trung bình, thang 0..255)

REF = {
    "add": "A {n} has been added.", "remove": "The {n} has been removed.",
    "color": "The {n} has changed color.", "move": "The {n} has been moved.",
    "size_up": "The {n} has become larger.", "size_down": "The {n} has become smaller.",
    "light_up": "The scene has become brighter.", "light_down": "The scene has become darker.",
    "same": "There is no difference between the two images.",
    "jpeg": "There is no difference between the two images.",
}


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--omnidiff-root", required=True)
    p.add_argument("--questions", nargs="+", required=True, help="file câu hỏi OmniDiff test (lấy ảnh nền + prompt)")
    p.add_argument("--objects-dir", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--n-per-cond", type=int, default=30)
    p.add_argument("--n-bg-per-split", type=int, default=60)
    p.add_argument("--seed", type=int, default=2026)
    p.add_argument("--sanity", action="store_true", help="chỉ sinh 1 cặp mỗi loại (mức giữa) + 2 đối chứng")
    return p.parse_args()


# ---------- tiện ích ảnh ----------
def resize_long(img, side=LONG_SIDE):
    s = side / max(img.shape[:2])
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)


def prepare_object(rgba, area_px, scale=1.0):
    """Phóng/thu vật để diện tích mặt nạ ≈ area_px (× scale theo chiều dài). Trả về (rgb float, alpha float)."""
    alpha0 = rgba[..., 3] > 127
    k = np.sqrt(area_px / max(alpha0.sum(), 1)) * scale
    h, w = rgba.shape[:2]
    nw, nh = max(4, int(round(w * k))), max(4, int(round(h * k)))
    interp = cv2.INTER_AREA if k < 1 else cv2.INTER_CUBIC
    rgb = cv2.resize(rgba[..., :3], (nw, nh), interpolation=interp).astype(np.float32)
    a = cv2.resize(rgba[..., 3], (nw, nh), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
    a = (a > 0.5).astype(np.float32)
    a = cv2.GaussianBlur(a, (0, 0), FEATHER_SIGMA)          # làm mềm mép để vật không lộ viền dán
    return rgb, a


def paste(bg, rgb, a, cx, cy):
    """Dán vật (tâm tại cx, cy). Trả về (ảnh uint8, alpha toàn ảnh) hoặc (None, None) nếu tràn ra ngoài."""
    H, W = bg.shape[:2]
    h, w = a.shape
    x0, y0 = int(round(cx - w / 2)), int(round(cy - h / 2))
    if x0 < MARGIN or y0 < MARGIN or x0 + w > W - MARGIN or y0 + h > H - MARGIN:
        return None, None
    out = bg.astype(np.float32).copy()
    reg = out[y0:y0 + h, x0:x0 + w]
    out[y0:y0 + h, x0:x0 + w] = reg * (1 - a[..., None]) + rgb * a[..., None]
    full = np.zeros((H, W), np.float32)
    full[y0:y0 + h, x0:x0 + w] = a
    return np.clip(out + 0.5, 0, 255).astype(np.uint8), full


def sample_center(rng, W, H, w, h):
    """Tâm ngẫu nhiên, ưu tiên nửa dưới ảnh (vật đặt trên mặt phẳng trông tự nhiên hơn)."""
    xlo, xhi = w / 2 + MARGIN, W - w / 2 - MARGIN
    ylo, yhi = max(h / 2 + MARGIN, 0.35 * H), H - h / 2 - MARGIN
    if ylo >= yhi:
        ylo = h / 2 + MARGIN
    if xlo >= xhi or ylo >= yhi:
        return None
    return float(rng.uniform(xlo, xhi)), float(rng.uniform(ylo, yhi))


def hue_shift(rgb, deg):
    hsv = cv2.cvtColor(np.clip(rgb, 0, 255).astype(np.uint8), cv2.COLOR_BGR2HSV).astype(np.int32)
    hsv[..., 0] = (hsv[..., 0] + int(round(deg / 2))) % 180          # OpenCV: H thang 0..179
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR).astype(np.float32)


def gamma(img, g):
    lut = np.clip(255.0 * (np.arange(256) / 255.0) ** g + 0.5, 0, 255).astype(np.uint8)
    return cv2.LUT(img, lut)


# ---------- sinh 1 cặp ----------
def make_pair(cond, lvl_idx, rng, bgs, objs, objs_color):
    """Trả về (A, B, meta) hoặc None nếu lần thử này không hợp lệ (gọi lại với rng tiếp theo)."""
    bi = int(rng.integers(len(bgs)))
    bg_name, bg = bgs[bi]
    H, W = bg.shape[:2]
    meta = {"bg_source": bg_name}
    if cond == "light":
        g = LEVELS["light"][lvl_idx]
        up = bool(rng.integers(2))
        B = gamma(bg, 1 / g if up else g)
        meta |= {"gamma": round(1 / g if up else g, 4), "direction": "brighter" if up else "darker",
                 "ref": REF["light_up" if up else "light_down"]}
        return bg.copy(), B, meta

    pool = objs_color if cond == "color" else objs
    o = pool.iloc[int(rng.integers(len(pool)))]
    rgba = o["_rgba"]
    meta |= {"obj_id": int(o["obj_id"]), "category": o["category"], "name": o["category"]}
    area_frac = LEVELS[cond][lvl_idx] if cond in ("add", "remove") else BASE_AREA[cond]
    rgb, a = prepare_object(rgba, area_frac * W * H)
    c = sample_center(rng, W, H, a.shape[1], a.shape[0])
    if c is None:
        return None
    withobj, am = paste(bg, rgb, a, *c)
    if withobj is None:
        return None
    meta |= {"area_frac_A": round(float(am.sum()) / (W * H), 4), "pos_A": [round(c[0]), round(c[1])]}
    n = o["category"]

    if cond == "add":
        meta |= {"area_frac_A": 0.0, "area_frac_B": round(float(am.sum()) / (W * H), 4),
                 "pos_A": None, "pos_B": [round(c[0]), round(c[1])], "ref": REF["add"].format(n=n)}
        return bg.copy(), withobj, meta
    if cond == "remove":
        meta |= {"area_frac_B": 0.0, "pos_B": None, "ref": REF["remove"].format(n=n)}
        return withobj, bg.copy(), meta
    if cond in ("same", "jpeg"):
        meta |= {"area_frac_B": meta["area_frac_A"], "pos_B": meta["pos_A"], "ref": REF[cond]}
        return withobj, withobj.copy(), meta
    if cond == "color":
        d = LEVELS["color"][lvl_idx] * (1 if rng.integers(2) else -1)
        B, _ = paste(bg, hue_shift(rgb, d), a, *c)
        meta |= {"hue_shift": d, "area_frac_B": meta["area_frac_A"], "pos_B": meta["pos_A"],
                 "ref": REF["color"].format(n=n)}
        return withobj, B, meta
    if cond == "move":
        dx = LEVELS["move"][lvl_idx] * W
        for sgn in ([1, -1] if rng.integers(2) else [-1, 1]):
            B, bm = paste(bg, rgb, a, c[0] + sgn * dx, c[1])
            if B is not None:
                meta |= {"shift_px": round(sgn * dx), "area_frac_B": round(float(bm.sum()) / (W * H), 4),
                         "pos_B": [round(c[0] + sgn * dx), round(c[1])], "ref": REF["move"].format(n=n)}
                return withobj, B, meta
        return None
    if cond == "size":
        s = LEVELS["size"][lvl_idx]
        rgb2, a2 = prepare_object(rgba, BASE_AREA["size"] * W * H, scale=s)
        # giữ đáy vật cố định (vật "đứng" tại chỗ khi to/nhỏ ra)
        cy2 = c[1] + a.shape[0] / 2 - a2.shape[0] / 2
        B, bm = paste(bg, rgb2, a2, c[0], cy2)
        if B is None:
            return None
        meta |= {"scale": s, "area_frac_B": round(float(bm.sum()) / (W * H), 4),
                 "pos_B": [round(c[0]), round(cy2)],
                 "ref": REF["size_up" if s > 1 else "size_down"].format(n=n)}
        return withobj, B, meta
    raise ValueError(cond)


def main():
    a = parse()
    rng0 = np.random.default_rng(a.seed)
    out = a.out
    for d in ["images", "masks"]:
        os.makedirs(os.path.join(out, d), exist_ok=True)

    # nền: ảnh A (duy nhất) của từng file câu hỏi OmniDiff test
    prompts, bgs = [], []
    for qf in a.questions:
        qs = json.load(open(qf))
        prompts += [q["conversations"][0]["value"] for q in qs]
        uniq = sorted({q["image"][0] for q in qs})
        pick = [uniq[i] for i in rng0.permutation(len(uniq))[:a.n_bg_per_split]]
        for rel in pick:
            img = cv2.imread(os.path.join(a.omnidiff_root, rel), cv2.IMREAD_COLOR)
            if img is not None:
                bgs.append((rel, resize_long(img)))
    prompt_pool = sorted(set(prompts))
    print(f"Ảnh nền: {len(bgs)} | prompt: {len(prompt_pool)} mẫu câu", flush=True)

    objs = pd.read_csv(os.path.join(a.objects_dir, "objects.csv"))
    objs["_rgba"] = [cv2.imread(os.path.join(a.objects_dir, f), cv2.IMREAD_UNCHANGED) for f in objs["file"]]
    objs = objs[objs["_rgba"].map(lambda x: x is not None and x.ndim == 3 and x.shape[2] == 4)].reset_index(drop=True)
    objs_color = objs[objs["mean_sat"] >= MIN_SAT_COLOR].reset_index(drop=True)
    print(f"Vật thể: {len(objs)} (đủ đậm màu cho 'đổi màu': {len(objs_color)})", flush=True)

    plan = []
    if a.sanity:
        plan = [(t, 1, 0) for t in ["add", "remove", "color", "move", "size", "light"]] + [("same", 0, 0), ("jpeg", 0, 0)]
    else:
        for t in LEVELS:
            for li in range(3):
                plan += [(t, li, k) for k in range(a.n_per_cond)]
        for t in ["same", "jpeg"]:
            plan += [(t, 0, k) for k in range(a.n_per_cond)]

    manifest, questions = [], []
    for idx, (cond, li, k) in enumerate(plan):
        pid = f"{cond}_L{li+1}_{k:02d}" if cond in LEVELS else f"{cond}_{k:02d}"
        seed = a.seed * 1000 + idx
        rng = np.random.default_rng(seed)
        res, tries = None, 0
        while res is None and tries < 200:
            res = make_pair(cond, li, rng, bgs, objs, objs_color)
            tries += 1
        if res is None:
            print("  bỏ qua (không đặt được vật):", pid); continue
        A, B, meta = res
        gt = np.any(A != B, axis=2)                      # GT trên ảnh chưa nén
        pa, pb = f"images/{pid}_A.png", f"images/{pid}_B.png"
        cv2.imwrite(os.path.join(out, pa), A)
        if cond == "jpeg":
            pb = f"images/{pid}_B.jpg"
            cv2.imwrite(os.path.join(out, pb), B, [cv2.IMWRITE_JPEG_QUALITY, JPEG_Q])
            gt[:] = False                                # không có thay đổi ngữ nghĩa
        else:
            cv2.imwrite(os.path.join(out, pb), B)
        pm = f"masks/{pid}.png"
        cv2.imwrite(os.path.join(out, pm), gt.astype(np.uint8) * 255)
        prompt = prompt_pool[int(rng.integers(len(prompt_pool)))]
        rec = {"pair_id": pid, "cond": cond, "level": (li + 1) if cond in LEVELS else 0,
               "level_value": LEVELS[cond][li] if cond in LEVELS else (JPEG_Q if cond == "jpeg" else None),
               "seed": seed, "tries": tries, "width": int(A.shape[1]), "height": int(A.shape[0]),
               "image_A": pa, "image_B": pb, "gt_mask": pm, "gt_frac": round(float(gt.mean()), 5),
               "prompt": prompt} | meta
        manifest.append(rec)
        questions.append({"id": pid, "image": [pa, pb], "conversations": [{"from": "human", "value": prompt}]})
        if (idx + 1) % 100 == 0:
            print(f"  {idx+1}/{len(plan)}", flush=True)

    with open(os.path.join(out, "manifest.jsonl"), "w") as f:
        for r in manifest:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    json.dump(questions, open(os.path.join(out, "questions.json"), "w"), indent=1)
    df = pd.DataFrame(manifest)
    print(f"\nĐã sinh {len(df)} cặp -> {out}")
    print(df.groupby(["cond", "level"]).size().unstack(fill_value=0).to_string())


if __name__ == "__main__":
    main()

**show_pairs.py** — Vẽ lưới kiểm tra A | B | GT

In [ ]:
%%writefile /mnt/m3diff/synth/code/show_pairs.py
"""Vẽ lưới kiểm tra: mỗi hàng 1 cặp — ảnh A | ảnh B | GT (A ≠ B) tô đỏ lên B."""
import argparse, json, os, sys
import cv2
import numpy as np
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from synth_common import load_manifest, load_pair, load_gt, TYPE_VI  # noqa: E402

p = argparse.ArgumentParser()
p.add_argument("--root", required=True); p.add_argument("--out", required=True)
p.add_argument("--n", type=int, default=0, help="0 = tất cả; >0 = chọn ngẫu nhiên n cặp")
p.add_argument("--captions", default=None, help="(tuỳ chọn) file câu trả lời để in kèm")
p.add_argument("--per-page", type=int, default=10)
a = p.parse_args()

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

recs = load_manifest(a.root)
if a.n:
    rng = np.random.default_rng(3)
    recs = [recs[i] for i in sorted(rng.permutation(len(recs))[:a.n])]
caps = {}
if a.captions and os.path.exists(a.captions):
    caps = {json.loads(l)["id"]: json.loads(l)["caption"] for l in open(a.captions) if l.strip()}
rgb = lambda x: cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
paths = []
for page in range(0, len(recs), a.per_page):
    chunk = recs[page:page + a.per_page]
    fig, axes = plt.subplots(len(chunk), 3, figsize=(15, 3.6 * len(chunk)), facecolor="#fcfcfb", squeeze=False)
    for row, r in zip(axes, chunk):
        A, B = load_pair(a.root, r)
        gt = load_gt(a.root, r)
        ov = rgb(B).copy()
        if gt is not None and gt.any():
            ov[gt] = (0.45 * ov[gt] + 0.55 * np.array([220, 38, 38])).astype(np.uint8)
        extra = {k: r[k] for k in ["category", "level_value", "hue_shift", "shift_px", "scale", "gamma", "direction"] if r.get(k) is not None}
        row[0].imshow(rgb(A)); row[0].set_title(f"{r['pair_id']} — {TYPE_VI[r['cond']]}  {extra}", loc="left", fontsize=9)
        row[1].imshow(rgb(B)); row[1].set_title("Ảnh B", loc="left", fontsize=9)
        row[2].imshow(ov); row[2].set_title(f"GT (A ≠ B) = {r['gt_frac']*100:.1f}% ảnh", loc="left", fontsize=9)
        if r["pair_id"] in caps:
            row[1].set_xlabel("M3Diff: " + caps[r["pair_id"]][:160], fontsize=8, wrap=True)
        for ax in row:
            ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    path = f"{a.out}_{page // a.per_page + 1}.png"
    fig.savefig(path, dpi=80, facecolor="#fcfcfb"); plt.close(fig)
    paths.append(path)
print("\n".join(paths))

**score_captions.py** — Chấm câu bằng bộ phân loại từ khoá

In [ ]:
%%writefile /mnt/m3diff/synth/code/score_captions.py
"""Chấm câu M3Diff sinh ra trên bộ test tổng hợp bằng bộ phân loại từ khoá.

3 chỉ số cho mỗi cặp có vật thể:
  - type_ok  : câu có nhắc đúng loại thay đổi (thêm/xoá/đổi màu/dời/đổi kích thước/đổi sáng)
  - obj_ok   : câu có nhắc đúng tên vật (kể cả từ đồng nghĩa)
  - joint_ok : cả hai
Đối chứng: false_alarm = câu có nhắc một thay đổi cụ thể (thêm/xoá/màu/dời/kích thước/sáng).

Chế độ --validate: đọc file đã được nhóm gán nhãn tay -> Precision/Recall/F1 của bộ phân loại từ khoá.
"""
import argparse, json, os, re, sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from synth_common import TYPES, CONTROLS, OBJECT_TYPES, TYPE_VI, load_manifest, wilson  # noqa: E402

COLORS = r"(red|orange|yellow|green|blue|purple|violet|pink|brown|black|white|gr[ae]y|cyan|magenta|teal|beige|golden|silver|maroon|navy|turquoise)"
PAT = {
    "add": r"\b(appear(s|ed|ing)?|added|adds?|adding|newly|a new|an new|introduc(ed|es)|emerg(ed|es)|now (present|visible|there|appears?|sits?|stands?)|there (is|are) now|has been placed|have been placed|been put)\b",
    "remove": r"\b(disappear(s|ed|ing)?|removed|remov(es|ing)|missing|no longer|gone|absent|taken away|vanish(ed|es)?|(is|are) not (present|visible)|been taken)\b",
    "color": r"\b(colou?r(s|ed)?|hue|tint)\b|\b(turned|became|become|becomes|now|changed to|painted)\s+(a\s+|an\s+)?(light\s+|dark\s+|bright\s+)?" + COLORS + r"\b|\bfrom\s+(a\s+)?(light\s+|dark\s+)?" + COLORS + r"\b[^.]{0,40}?\bto\s+(a\s+)?(light\s+|dark\s+)?" + COLORS + r"\b",
    "move": r"\b(mov(ed|es|ing)|shift(ed|s|ing)?|relocat(ed|es|ing)|reposition(ed|s)?|different (position|location|place|spot)|(position|location) (has |have )?changed|changed (its |their )?(position|location)|slid|displaced|moved to)\b",
    "size": r"\b(larger|bigger|smaller|enlarged|shrunk|shrank|shrink(s|ing)?|grown|grew|expanded|increased in size|decreased in size|reduced in size|size|scaled)\b",
    "light": r"\b(brighter|darker|dimmer|dimmed|brightness|lighting|illuminat\w*|exposure|lighter|light (level|intensity|condition)s?|more light|less light)\b",
}
VIEW = r"\b(camera|viewpoint|perspective|angle|zoom(ed|s|ing)?|vantage|field of view)\b"
NOCHANGE = r"\b(no (visible |noticeable |significant |apparent |obvious )?(changes?|differences?|modifications?|alterations?)|(are|look|appear|remain) (identical|the same|exactly the same)|nothing (has )?changed|identical)\b"
SYN = {
    "bottle": ["bottle"], "cup": ["cup", "mug", "glass"], "bowl": ["bowl", "dish"], "vase": ["vase"],
    "chair": ["chair", "seat", "stool"], "potted plant": ["plant", "flowerpot", "potted", "flower", "pot"],
    "teddy bear": ["teddy", "bear", "stuffed", "plush", "toy"], "backpack": ["backpack", "bag", "rucksack", "knapsack"],
    "umbrella": ["umbrella", "parasol"], "suitcase": ["suitcase", "luggage", "bag", "case", "trolley"],
    "laptop": ["laptop", "computer", "notebook"], "book": ["book", "notebook"], "clock": ["clock"],
    "dog": ["dog", "puppy"], "cat": ["cat", "kitten"], "bicycle": ["bicycle", "bike"],
    "car": ["car", "vehicle", "automobile", "sedan", "suv"], "apple": ["apple", "fruit"],
    "banana": ["banana", "fruit"], "handbag": ["handbag", "bag", "purse"],
}
RX = {k: re.compile(v, re.I) for k, v in PAT.items()}
RX_VIEW, RX_NO = re.compile(VIEW, re.I), re.compile(NOCHANGE, re.I)
TRUE_TYPE = {t: t for t in TYPES}


def classify(caption):
    sents = re.split(r"(?<=[.!?;])\s+", caption)
    lab = {k: bool(r.search(caption)) for k, r in RX.items()}
    # 'shifted/moved' trong câu nói về máy ảnh là đổi góc nhìn, không phải dời vật
    lab["move"] = any(RX["move"].search(s) and not RX_VIEW.search(s) for s in sents)
    return lab | {
        "view": bool(RX_VIEW.search(caption)), "nochange": bool(RX_NO.search(caption))}


def mentions_object(caption, category):
    if not isinstance(category, str):
        return False
    words = SYN.get(category, [category])
    return any(re.search(r"\b" + re.escape(w) + r"(s|es)?\b", caption, re.I) for w in words)


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--root", required=True, help="thư mục bộ tổng hợp (có manifest.jsonl)")
    p.add_argument("--answers", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--n-manual", type=int, default=20, help="số câu mỗi loại đưa ra để gán nhãn tay")
    p.add_argument("--validate", default=None, help="file CSV đã gán nhãn tay -> P/R/F1")
    return p.parse_args()


def score(a):
    man = pd.DataFrame(load_manifest(a.root))
    ans = {}
    for l in open(a.answers):
        if l.strip():
            r = json.loads(l)
            ans[r["id"]] = r["caption"]
    man["caption"] = man["pair_id"].map(ans)
    miss = man["caption"].isna().sum()
    if miss:
        print(f"!! Thiếu câu trả lời cho {miss} cặp — chỉ chấm các cặp đã có")
    df = man.dropna(subset=["caption"]).copy()
    lab = pd.DataFrame([classify(c) for c in df["caption"]], index=df.index)
    df = pd.concat([df, lab.add_prefix("m_")], axis=1)
    df["n_words"] = df["caption"].str.split().map(len)
    df["type_ok"] = [row[f"m_{row['cond']}"] if row["cond"] in TYPES else np.nan for _, row in df.iterrows()]
    df["obj_ok"] = [mentions_object(row["caption"], row["category"]) if row["cond"] in OBJECT_TYPES else np.nan
                    for _, row in df.iterrows()]
    df["joint_ok"] = [(row["type_ok"] and row["obj_ok"]) if row["cond"] in OBJECT_TYPES else np.nan
                      for _, row in df.iterrows()]
    df["false_alarm"] = [any(row[f"m_{t}"] for t in TYPES) if row["cond"] in CONTROLS else np.nan
                         for _, row in df.iterrows()]
    os.makedirs(a.out, exist_ok=True)
    keep = [c for c in df.columns if not c.startswith("_")]
    df[keep].to_csv(os.path.join(a.out, "per_pair.csv"), index=False)

    # ---- bảng theo loại × mức ----
    rows = []
    for (cond, lvl), g in df[df["cond"].isin(TYPES)].groupby(["cond", "level"]):
        r = {"cond": cond, "level": lvl, "level_value": g["level_value"].iloc[0], "n": len(g)}
        for m in ["type_ok", "obj_ok", "joint_ok"]:
            v = g[m].dropna()
            if len(v):
                p, lo, hi = wilson(int(v.sum()), len(v))
                r[m] = p; r[m + "_lo"] = lo; r[m + "_hi"] = hi
        rows.append(r)
    summ = pd.DataFrame(rows)
    order = {t: i for i, t in enumerate(TYPES)}
    summ = summ.sort_values(["cond", "level"], key=lambda s: s.map(order) if s.name == "cond" else s)
    summ.to_csv(os.path.join(a.out, "summary_by_level.csv"), index=False)

    print("\n== Tỉ lệ đúng theo loại × mức (%, khoảng tin cậy 95% Wilson) ==")
    print(f"{'Loại':<16}{'Mức':>4}{'Giá trị':>9}{'n':>4}   {'Đúng loại':<17}{'Đúng vật':<17}{'Đúng cả hai':<17}")
    fmt = lambda r, m: (f"{r[m]*100:5.1f} [{r[m+'_lo']*100:3.0f}–{r[m+'_hi']*100:3.0f}]"
                        if m in r and pd.notna(r.get(m)) else "   —")
    for _, r in summ.iterrows():
        print(f"{TYPE_VI[r['cond']]:<16}{r['level']:>4}{r['level_value']:>9}{r['n']:>4}   "
              f"{fmt(r,'type_ok'):<17}{fmt(r,'obj_ok'):<17}{fmt(r,'joint_ok'):<17}")

    print("\n== Tổng theo loại (gộp 3 mức) ==")
    for t in TYPES:
        g = df[df["cond"] == t]
        if len(g):
            s = f"{TYPE_VI[t]:<16} n={len(g):<4} đúng loại {g['type_ok'].mean()*100:5.1f}%"
            if t in OBJECT_TYPES:
                s += f" | đúng vật {g['obj_ok'].mean()*100:5.1f}% | cả hai {g['joint_ok'].mean()*100:5.1f}%"
            print(s)

    print("\n== Đối chứng (không có thay đổi ngữ nghĩa) ==")
    for t in CONTROLS:
        g = df[df["cond"] == t]
        if len(g):
            p, lo, hi = wilson(int(g["false_alarm"].sum()), len(g))
            print(f"{TYPE_VI[t]:<24} n={len(g):<4} báo thay đổi giả: {p*100:5.1f}% [{lo*100:.0f}–{hi*100:.0f}] | "
                  f"nói 'không đổi': {g['m_nochange'].mean()*100:5.1f}% | nhắc góc máy: {g['m_view'].mean()*100:5.1f}%")

    # ---- ma trận: loại thật -> các loại được nhắc tới ----
    cols = TYPES + ["view", "nochange"]
    mat = df.groupby("cond")[[f"m_{c}" for c in cols]].mean() * 100
    mat.columns = cols
    mat = mat.reindex([t for t in TYPES + CONTROLS if t in mat.index])
    mat.to_csv(os.path.join(a.out, "mention_matrix.csv"))
    print("\n== Ma trận nhắc tới (% câu của mỗi loại thật có nhắc tới từng loại) ==")
    print(mat.round(0).astype(int).to_string())

    plots(df, summ, mat, a.out)
    export_manual(df, a)
    return df


def plots(df, summ, mat, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ink, muted, grid = "#0b0b0b", "#52525b", "#e4e4e7"
    pal = ["#2a78d6", "#1f9d74", "#d97706", "#c2415d", "#7c5cd6", "#4a5568"]
    xlabels = {"add": ["1%", "4%", "10%"], "remove": ["1%", "4%", "10%"], "color": ["30°", "60°", "120°"],
               "move": ["10%", "25%", "50%"], "size": ["×0.6", "×1.5", "×2"], "light": ["γ1.25", "γ1.6", "γ2.2"]}
    fig, axes = plt.subplots(1, 6, figsize=(22, 4.2), sharey=True, facecolor="#fcfcfb")
    for ax, t, c in zip(axes, TYPES, pal):
        g = summ[summ["cond"] == t]
        if not len(g):
            ax.axis("off"); continue
        x = np.arange(len(g))
        ax.plot(x, g["type_ok"] * 100, "-o", color=c, lw=2, label="Đúng loại")
        ax.fill_between(x, g["type_ok_lo"] * 100, g["type_ok_hi"] * 100, color=c, alpha=0.12, lw=0)
        if t in OBJECT_TYPES:
            ax.plot(x, g["joint_ok"] * 100, "--s", color=c, lw=1.5, alpha=0.7, label="Đúng loại + đúng vật")
        ax.set_xticks(x); ax.set_xticklabels(xlabels[t][:len(g)], color=muted)
        ax.set_title(TYPE_VI[t], loc="left", fontsize=11, color=ink)
        ax.set_ylim(0, 100); ax.grid(axis="y", color=grid, lw=0.8)
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)
        ax.tick_params(colors=muted)
        ax.legend(frameon=False, fontsize=8, loc="lower right")
    axes[0].set_ylabel("% cặp", color=muted)
    fig.suptitle("Tỉ lệ M³Diff mô tả đúng theo mức độ thay đổi (vùng mờ: khoảng tin cậy 95%)",
                 x=0.01, ha="left", fontsize=12, color=ink)
    fig.tight_layout(); fig.savefig(os.path.join(out, "detection_by_level.png"), dpi=120, facecolor="#fcfcfb")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 5), facecolor="#fcfcfb")
    im = ax.imshow(mat.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels(["thêm", "xoá", "màu", "dời", "kích thước", "sáng", "góc máy", "'không đổi'"],
                       rotation=30, ha="right", color=muted)
    ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels([TYPE_VI[t] for t in mat.index], color=muted)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9, color="white" if v > 55 else ink)
    ax.set_xlabel("Câu model sinh ra có nhắc tới", color=muted); ax.set_ylabel("Loại thay đổi thật", color=muted)
    ax.set_title("Ma trận nhắc tới (%)", loc="left", color=ink)
    fig.colorbar(im, ax=ax, fraction=0.03)
    fig.tight_layout(); fig.savefig(os.path.join(out, "mention_matrix.png"), dpi=120, facecolor="#fcfcfb")
    plt.close(fig)


def export_manual(df, a):
    """Xuất ~20 câu/loại để nhóm gán nhãn tay. KHÔNG kèm nhãn tự động (tránh thiên lệch)."""
    rng = np.random.default_rng(7)
    parts = []
    for t in TYPES:
        g = df[df["cond"] == t]
        if len(g):
            parts.append(g.iloc[rng.permutation(len(g))[:a.n_manual]])
    if not parts:
        return
    s = pd.concat(parts)
    s = s.iloc[rng.permutation(len(s))]                       # xáo để người gán không biết loại thật
    out = s[["pair_id", "category", "caption"]].copy()
    for c in ["add", "remove", "color", "move", "size", "light", "object"]:
        out["tay_" + c] = ""
    path = os.path.join(a.out, "manual_check.csv")
    out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\nĐã xuất {len(out)} câu để gán nhãn tay: {path}")


def validate(a):
    man = pd.read_csv(a.validate, encoding="utf-8-sig")
    auto = pd.read_csv(os.path.join(a.out, "per_pair.csv")).set_index("pair_id")
    rows = []
    for c in ["add", "remove", "color", "move", "size", "light", "object"]:
        col = "tay_" + c
        v = pd.to_numeric(man[col], errors="coerce")
        m = man[v.isin([0, 1])]
        if c == "object":
            m = m[m["category"].map(lambda x: isinstance(x, str))]     # loại 'đổi sáng' không có vật
        y = pd.to_numeric(m[col]).astype(int).values
        if c == "object":
            yhat = np.array([mentions_object(r["caption"], r["category"]) for _, r in m.iterrows()], int)
        else:
            yhat = auto.loc[m["pair_id"], f"m_{c}"].astype(int).values
        tp, fp, fn = int(((y == 1) & (yhat == 1)).sum()), int(((y == 0) & (yhat == 1)).sum()), int(((y == 1) & (yhat == 0)).sum())
        prec = tp / (tp + fp) if tp + fp else float("nan")
        rec = tp / (tp + fn) if tp + fn else float("nan")
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else float("nan")
        rows.append({"nhãn": c, "n": len(m), "dương (tay)": int(y.sum()), "precision": prec, "recall": rec, "f1": f1,
                     "accuracy": float((y == yhat).mean()) if len(m) else float("nan")})
    r = pd.DataFrame(rows)
    r.to_csv(os.path.join(a.out, "classifier_validation.csv"), index=False)
    print("== Độ chính xác của bộ phân loại từ khoá so với nhãn tay ==")
    print(r.round(3).to_string(index=False))


if __name__ == "__main__":
    args = parse()
    validate(args) if args.validate else score(args)

## Bước 6 — Lấy vật thể từ COCO (~5–10 phút, chỉ chạy lần đầu)
20 lớp dễ gọi tên (cup, bottle, chair, dog, car…), tối đa 15 vật/lớp. Lọc bỏ vật bị che, chạm mép ảnh, nhiều mảnh, quá nhỏ.
Tải annotations COCO (~250MB) và chỉ những ảnh chứa vật được chọn.

In [ ]:
from IPython.display import Image, display
import glob
OBJ = f"{SYN}/objects"
if not os.path.exists(f"{OBJ}/objects.csv"):
    !{PY} {CODE}/prep_coco_objects.py --cache {SYN}/coco_cache --out {OBJ} --per-cat 15
else:
    print("Đã có vật thể:", OBJ)
display(Image(f"{OBJ}/objects_sheet.png"))

Xem ảnh: vật phải được cắt gọn, không dính người/vật khác. `S` = độ đậm màu (vật S ≥ 70 mới dùng cho loại 'đổi màu').

## Bước 7 — Kiểm tra nhanh trên 8 cặp cố định (bắt buộc trước khi chạy 600 cặp)
1 cặp mỗi loại (mức giữa) + 2 đối chứng. Kiểm tra bằng mắt 3 điều:
1. Ảnh A/B đúng loại thay đổi ghi ở tiêu đề.
2. GT (tô đỏ) nằm đúng chỗ thay đổi; đối chứng GT = 0%; đổi sáng GT ≈ 100%.
3. Model chạy được và câu sinh ra có nghĩa.

(Kiểm tra heatmap và log-xác suất nằm ở đầu Notebook 2 và 3.)

In [ ]:
SAN = f"{SYN}/sanity"
!{PY} {CODE}/make_synth.py --omnidiff-root {REPO}/datasets/OmniDiff --questions {QFILES} \
  --objects-dir {OBJ} --out {SAN} --sanity

san_ans = f"{SAN}/answers.jsonl"
if os.path.exists(san_ans):
    os.remove(san_ans)
%cd {REPO}
!PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py \
  --model-path {CKPT} --question-file {SAN}/questions.json --answers-file {san_ans} \
  --image-folder {SAN} --num-chunks 1 --chunk-idx 0 --temperature 0 --torch-dtype {DTYPE} --no-flash-attention

!{PY} {CODE}/show_pairs.py --root {SAN} --out {SAN}/grid --captions {san_ans} --per-page 8
display(Image(f"{SAN}/grid_1.png"))
for l in open(san_ans):
    r = json.loads(l); print(f"\n[{r['id']}] {r['caption']}")

## Bước 8 — Sinh 600 cặp (~3–5 phút, không dùng GPU)
Sinh lại từ đầu thì đặt `REGEN = True` (cùng seed sẽ ra đúng bộ cũ).

In [ ]:
REGEN = False
if REGEN or not os.path.exists(f"{DATA}/manifest.jsonl"):
    !{PY} {CODE}/make_synth.py --omnidiff-root {REPO}/datasets/OmniDiff --questions {QFILES} \
      --objects-dir {OBJ} --out {DATA} --n-per-cond 30
else:
    print("Đã có bộ dữ liệu:", DATA)

# Lưới 30 cặp ngẫu nhiên để kiểm tra chất lượng
!{PY} {CODE}/show_pairs.py --root {DATA} --out {RES}/quality_grid --n 30 --per-page 10
for p in sorted(glob.glob(f"{RES}/quality_grid_*.png")):
    display(Image(p))m

## Bước 9 — Chạy M³Diff trên 600 cặp (~30 phút, chạy tiếp được nếu bị ngắt)

In [ ]:
def count_done(fp):
    if not os.path.exists(fp):
        return 0
    n = 0
    for l in open(fp):
        try:
            json.loads(l); n += 1
        except Exception:
            pass
    return n

ANS = f"{DATA}/answers.jsonl"
N_TOTAL = len(json.load(open(f"{DATA}/questions.json")))
print(f"Đã có: {count_done(ANS)}/{N_TOTAL}")
if count_done(ANS) < N_TOTAL:
    %cd {REPO}
    !PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py \
      --model-path {CKPT} --question-file {DATA}/questions.json --answers-file {ANS} \
      --image-folder {DATA} --num-chunks 1 --chunk-idx 0 --temperature 0 --torch-dtype {DTYPE} --no-flash-attention
print(f"Hiện có: {count_done(ANS)}/{N_TOTAL}")

## Bước 10 — Chấm câu (~1 phút)

In [ ]:
!{PY} {CODE}/score_captions.py --root {DATA} --answers {ANS} --out {RES} --n-manual 20
display(Image(f"{RES}/detection_by_level.png"))
display(Image(f"{RES}/mention_matrix.png"))

Cách đọc:
- **Đúng loại / đúng vật / đúng cả hai**: % cặp mà câu model sinh ra nhắc đúng loại thay đổi / đúng tên vật / cả hai. Trong ngoặc là khoảng tin cậy 95%.
- **Đối chứng**: "báo thay đổi giả" = model bịa ra một thay đổi dù 2 ảnh giống nhau (A == B) hoặc chỉ khác do nén JPEG.
- **Ma trận nhắc tới**: hàng = loại thay đổi thật, cột = loại mà câu có nhắc tới. Đường chéo càng đậm càng tốt; ô ngoài đường chéo cho biết model hay nhầm sang loại nào.

## Bước 11 — Kiểm tra bộ chấm bằng nhãn tay (~1 giờ đọc, làm chung cả nhóm)
Bộ chấm dùng từ khoá nên có thể sai. Cần đo xem nó sai bao nhiêu:
1. Vào **Files** → `/mnt/m3diff/synth/results/manual_check.csv` → tải về, mở bằng Excel (120 câu, đã xáo thứ tự, không ghi loại thật).
2. Với mỗi câu, điền **1/0** vào các cột `tay_add, tay_remove, tay_color, tay_move, tay_size, tay_light`: câu **có nhắc tới** kiểu thay đổi đó không (kể cả nhắc sai vật). Cột `tay_object`: câu có nhắc đúng vật ghi ở cột `category` không.
3. Lưu lại (giữ định dạng CSV), tải lên cùng thư mục với tên **`manual_filled.csv`**, rồi chạy cell dưới.

In [ ]:
FILLED = f"{RES}/manual_filled.csv"
if os.path.exists(FILLED):
    !{PY} {CODE}/score_captions.py --root {DATA} --answers {ANS} --out {RES} --validate {FILLED}
else:
    print("Chưa có", FILLED)

## Kết quả lưu ở `/mnt/m3diff/synth/`
- `data/`: 600 cặp ảnh, `masks/` (ground truth), `manifest.jsonl` (tham số từng cặp, có seed), `questions.json`, `answers.jsonl`
- `results/`: `per_pair.csv`, `summary_by_level.csv`, `mention_matrix.csv`, 2 biểu đồ, `quality_grid_*.png`, `classifier_validation.csv`
- `code/`: các script (Notebook 2, 3 dùng lại)

Xong nhớ bấm **Terminate**.